# LMTA ALMP XML Export

**Notebook engine:** PySpark.

This notebook uses the PySpark runtime only for Microsoft Fabric integration and `notebookutils`. It does not use Spark DataFrames. Warehouse access, XML construction, XSD validation, and Lakehouse file output are ordinary Python.

## Install Packages

In [ ]:
# Run only once
# %pip install lxml pyodbc

StatementMeta(, bea9d60a-cb2e-4201-98d7-33c9dcd0fcdb, 3, Finished, Available, Finished, False)

If Fabric restarts the session after package installation, continue with the next cell.


## Import Modules

In [10]:
from pathlib import Path
import sys
import importlib
import json

MODULE_DIR = Path("/lakehouse/default/Files/LMTA/code/python")
MODULE_PATH = MODULE_DIR / "lmta_almp_export.py"

if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Upload the new module to: {MODULE_PATH}")

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

sys.modules.pop("lmta_almp_export", None)
importlib.invalidate_caches()
import lmta_almp_export

print("Loaded from:", lmta_almp_export.__file__)
print("Module version:", lmta_almp_export.MODULE_VERSION)
print("ExportConfig available:", hasattr(lmta_almp_export, "ExportConfig"))
print("test_connection available:", hasattr(lmta_almp_export, "test_connection"))
print("run_export available:", hasattr(lmta_almp_export, "run_export"))

ExportConfig = lmta_almp_export.ExportConfig
test_connection = lmta_almp_export.test_connection
run_export = lmta_almp_export.run_export


StatementMeta(, bea9d60a-cb2e-4201-98d7-33c9dcd0fcdb, 12, Finished, Available, Finished, False)

Loaded from: /lakehouse/default/Files/LMTA/code/python/lmta_almp_export.py
Module version: 2026.11.07-clean-4
ExportConfig available: True
test_connection available: True
run_export available: True


## Create Configuration

In [ ]:
# EDIT THESE VALUES ONLY
SUBMISSION_ID = 1001
SQL_SERVER = "esfe5ty3mtjebec6tiziwzcpvm-efk4ghrfe6uuzkqv3r573kj6be.datawarehouse.fabric.microsoft.com"
SQL_DATABASE = "LMTA"
PROGRAM_TYPE = "LMDA"       # LMDA or WDA
SEQUENCE_NUMBER = 1
MAIN_XSD_NAME = "ALMP.xsd"

# Map column names in the database tables to the order of elements in the XML output
# which is defined in the XSD as <xsd:sequence>

ELEMENT_ORDER = {
    "submission": ["SchemaVersion", "agreementHolderName", "sysComment"],
    "sys_action": [
        "socialInsuranceNumber", "clientID", "agreementNumber",
        "actionPlanStartDate", "interventionStartDate", "interventionCode",
        "actionPlanID", "interventionID", "systemAction", "sysComment",
    ],
    "client": [
        "clientID", "socialInsuranceNumber", "lastName", "firstName",
        "dateOfBirth", "emailAddress", "gender", "maritalStatus",
        "numberOfDependantChildren", "languageSpoken", "disability",
        "indigenousIdentity", "immigrant", "immigrationYear", "visibleMinority",
        "streetAddress", "municipality", "province", "postalZIPCode",
        "contactPhoneNumber", "addressSysComment", "sysComment",
    ],
    "action_plan": [
        "actionPlanID", "agreementNumber", "actionPlanFundingSource",
        "educationLevel", "educationLevelComment", "provincialOfficeCode",
        "actionPlanPreviousEmployment", "actionPlanPreviousEmployerName",
        "actionPlanPreviousMultipleEmployment", "actionPlanPreviousEmploymentNOC",
        "actionPlanPreviousEmploymentNOCVersion", "actionPlanPreviousEmploymentNAICS",
        "actionPlanPreviousEmploymentNAICSVersion", "actionPlanPreviousEmploymentDate",
        "actionPlanPreviousEmploymentPayRate", "actionPlanPreviousEmploymentHours",
        "actionPlanTargetNAICS", "actionPlanTargetNAICSVersion",
        "actionPlanPrecariousEmployment", "actionPlanStartDate", "actionPlanEndDate",
        "actionPlanResultCode", "actionPlanResultDate", "sysComment",
    ],
    "followup": [
        "actionPlanFollowUpID", "actionPlanFollowUpDate",
        "actionPlanFollowUpResultCode", "actionPlanFollowUpResultDate",
        "actionPlanFollowUpEmployerName", "actionPlanFollowUpMultipleEmployment",
        "actionPlanFollowUpNOC", "actionPlanFollowUpNOCVersion",
        "actionPlanFollowUpNAICS", "actionPlanFollowUpNAICSVersion",
        "actionPlanFollowUpEmploymentPayRate", "actionPlanFollowUpEmploymentHours",
        "sysComment",
    ],
    "intervention": [
        "interventionID", "interventionTitle", "interventionCode",
        "interventionStartDate", "interventionEndDate", "interventionOutcome",
        "interventionRelatedNOC", "interventionRelatedNOCVersion",
        "interventionLanguageOfService", "interventionEssentialSkills",
        "interventionCredential", "sysComment",
    ],
    "support": [
        "interventionCost", "interventionCostCategory",
        "interventionCostCategoryComment",
    ],
}

config = ExportConfig(
    submission_id=SUBMISSION_ID,
    sql_server=SQL_SERVER,
    sql_database=SQL_DATABASE,
    program_type=PROGRAM_TYPE,
    sequence_number=SEQUENCE_NUMBER,
    main_xsd_name=MAIN_XSD_NAME,
    table_element_order=ELEMENT_ORDER,
)

print("Configuration created for submission:", config.submission_id)


StatementMeta(, bea9d60a-cb2e-4201-98d7-33c9dcd0fcdb, 13, Finished, Available, Finished, False)

Configuration created for submission: 1001


## Test Connection

In [12]:
# Run this before XML generation.
connection_result = test_connection(config)
print(json.dumps(connection_result, indent=2, default=str))


StatementMeta(, bea9d60a-cb2e-4201-98d7-33c9dcd0fcdb, 14, Finished, Available, Finished, False)

{
  "connected": true,
  "driver": "libmsodbcsql-18.4.so.1.1",
  "database": "LMTA",
  "latest_submission_id": 1001
}


## Run Export
**Required ICT_DL1 locations**

- Module: `Files/LMTA/code/python/lmta_almp_export.py`
- Main XSD: `Files/LMTA/reference/xsd/current/ALMP.xsd`
- Imported XSDs: same `current` folder
- Valid XML: `Files/LMTA/output/xml/valid/`
- Invalid XML: `Files/LMTA/output/xml/invalid/`
- Validation reports: `Files/LMTA/output/validation/`

In [ ]:
# Run only after the connection test succeeds and SUBMISSION_ID exists.
# Return validation reports instead of raising so the report cell can display errors.
config.fail_on_validation_error = False
result = None
result = run_export(config)
print(json.dumps(result, indent=2, default=str))